# PROFHiT on the paper's own bundled datasets (Kaggle GPU runner)

Before running: in the **Settings** panel on the right, set **Accelerator → GPU T4 x2** (or P100) and **Internet → Off** (not needed — these datasets are already committed to the repo, no external download required).

This clones `math0110/PROFHiT`, trains PROFHiT on `labour`, `tourismsmall`, `tourismlarge`, `traffic`, `wiki2`, and `m5` (a stratified 420-leaf-series subsample of the full 30,490-series M5 competition data — see `data/m5/import_m5.py` for how it was built), and writes one JSON per dataset to `results/`, with metrics broken out **per hierarchy level** as well as overall. To run this unattended (survives closing the browser tab), use **Save Version → Save & Run All (Commit)** instead of running cells interactively — the output files will appear under the notebook's Output tab once it finishes.

This replaces the earlier `kaggle_train_hf.ipynb` benchmark (prison/tourism/police/m5 from the zaai-ai HF dataset) — these are the datasets that actually matter for the report.

In [ ]:
!git clone https://github.com/math0110/PROFHiT.git
%cd PROFHiT

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check notebook Settings > Accelerator")

## Train all six datasets

`m5` and `tourismlarge` have the most nodes/timesteps of this batch and will take the longest — each is wrapped in its own `try/except` so one running long or failing doesn't lose results already saved for the others.

In [ ]:
import json
import subprocess
import time

DATASETS = ["labour", "tourismsmall", "traffic", "wiki2", "tourismlarge", "m5"]
COMMON_ARGS = [
    "--device", "cuda",
    "--seeds", "1",
    "--pretrain-epochs", "20",
    "--epochs", "40",
    "--eval-samples", "100",
    "--out-dir", "results",
]

run_summary = {}
for name in DATASETS:
    print(f"\n{'='*60}\nStarting {name}\n{'='*60}")
    t0 = time.time()
    try:
        subprocess.run(["python", "train_tags.py", "--dataset", name] + COMMON_ARGS, check=True)
        run_summary[name] = {"status": "ok", "elapsed_sec": time.time() - t0}
    except subprocess.CalledProcessError as e:
        run_summary[name] = {"status": "failed", "error": str(e), "elapsed_sec": time.time() - t0}
        print(f"!! {name} failed, continuing with remaining datasets")

print("\nRun summary:")
print(json.dumps(run_summary, indent=2))

## Inspect results

Each result file nests metrics under `by_hierarchy -> <hierarchy name> -> "overall" | "level_<k>"`. Most datasets have one hierarchy (named `"tree"`); `tourismlarge` has two (`"geography"` and `"purpose"`), matching its crossed structure.

In [ ]:
import json
from pathlib import Path

for f in sorted(Path("results").glob("*.json")):
    d = json.load(open(f))
    print(f"\n=== {f.name} ===")
    for run in d["runs"]:
        print(f"seed={run['seed']}  dce={run['dce']:.4f}  n_nodes={run['num_nodes']}")
        for hier_name, levels in run["by_hierarchy"].items():
            print(f"  hierarchy: {hier_name}")
            for level_key, m in levels.items():
                print(
                    f"    {level_key:10s} rmse={m['rmse']:.3f} wape={m['wape']:.3f} "
                    f"crps={m['crps']:.3f} ls={m['log_score']:.3f} cs={m['calibration_score']:.3f}"
                )